# Week 5 · Day 5 — Mini-Project: Data Analysis for Marketing Strategy

**Dataset:** US Superstore (Excel `.xls`). **Goal:** Use area, customer, category and time-series analysis to identify marketing priorities. All numerical findings below are calculated when you run the notebook.

**Instructions:** Run cells from top to bottom. The first data cell downloads the file automatically; if download fails, manually save `US Superstore data.xls` beside this notebook. Charts are displayed and also saved under `results/`.

In [ ]:
# Run once in a notebook if dependencies are missing:
# %pip install pandas numpy matplotlib seaborn xlrd ipywidgets openpyxl
import io
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
ROOT = Path.cwd()
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
DATA_URL = "https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%205%20-%20Data%20Processing/W5D5%20-%20Mini-project%20-%20bis/US%20Superstore%20data.xls"

def save_chart(name):
    plt.tight_layout()
    plt.savefig(OUT / name, dpi=160, bbox_inches="tight")
    plt.show()

def money(x):
    return f"${x:,.2f}"


## 1. Load, assess and clean the data

Remove **exact duplicate records** to prevent double counting. Drop rows missing essential identifiers or numeric measures rather than inventing sales or profits. Preserve missing postal codes because they are not needed for state/city analysis. Parse dates explicitly and create a weighted profit margin: total profit / total sales.

In [ ]:
local_files = [ROOT / "US Superstore data.xls", ROOT / "Sample - Superstore.xls", ROOT / "superstore.xls"]
source = next((p for p in local_files if p.exists()), None)
if source is None:
    print("Downloading US Superstore dataset...")
    try:
        request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=40) as response:
            raw = response.read()
        source = ROOT / "US Superstore data.xls"
        source.write_bytes(raw)
    except Exception as exc:
        raise FileNotFoundError("Download failed. Save 'US Superstore data.xls' beside this notebook and rerun.") from exc
print("Loading:", source)
try:
    raw_df = pd.read_excel(source, engine="xlrd")
except Exception:
    raw_df = pd.read_excel(source)  # Handles files with xlsx content despite their extension
print("Original shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
print("Exact duplicate rows:", raw_df.duplicated().sum())
display(raw_df.head())
display(raw_df.isna().sum().rename("Missing values").to_frame())


In [ ]:
df = raw_df.drop_duplicates().copy()
required = ["Order Date", "State", "City", "Customer ID", "Customer Name", "Category", "Sales", "Profit"]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise KeyError(f"Required columns missing: {missing_columns}")
for c in ["Sales", "Profit", "Discount", "Quantity"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
for c in ["Order Date", "Ship Date"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce", dayfirst=False)
rows_before = len(df)
df = df.dropna(subset=required).copy()
df = df.loc[df["Sales"] > 0].copy()
for c in ["State", "City", "Customer ID", "Customer Name", "Category"]:
    df[c] = df[c].astype(str).str.strip()
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.to_period("M")
df["Profit Margin (%)"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"] * 100, np.nan)
print("Cleaned shape:", df.shape, "| Rows removed after deduplication:", len(raw_df) - len(df))
print("Total sales:", money(df["Sales"].sum()), "| Total profit:", money(df["Profit"].sum()))
print("Overall profit margin:", f'{df["Profit"].sum()/df["Sales"].sum()*100:.2f}%')
display(df[["Order Date", "State", "City", "Customer Name", "Sales", "Profit", "Profit Margin (%)"]].head())
df.to_csv(OUT / "cleaned_superstore.csv", index=False)


## 2. Which states have the most sales?

Group by state, rank by sales, and compare **total profit** and **aggregate profit margin** to avoid treating revenue as profitability.

In [ ]:
states = df.groupby("State", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique") if "Order ID" in df.columns else ("Sales", "size"))
states["Margin (%)"] = states["Profit"] / states["Sales"] * 100
states = states.sort_values("Sales", ascending=False).reset_index(drop=True)
display(states.head(20))
states.to_csv(OUT / "state_performance.csv", index=False)
plt.figure(figsize=(11, 7))
sns.barplot(data=states.head(15), x="Sales", y="State", color="steelblue")
plt.title("Top 15 States by Total Sales")
plt.xlabel("Sales ($)")
plt.ylabel("State")
save_chart("01_top_states_sales.png")


## 3. New York vs. California — sales and profit

Compare totals and profit margins, rather than using sales alone as the measure of success.

In [ ]:
ny_ca = states.loc[states["State"].isin(["New York", "California"])].copy()
display(ny_ca)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=ny_ca, x="State", y="Sales", ax=axes[0], color="cornflowerblue")
sns.barplot(data=ny_ca, x="State", y="Profit", ax=axes[1], color="seagreen")
axes[0].set_title("Total Sales: NY vs California")
axes[1].set_title("Total Profit: NY vs California")
for ax in axes: ax.tick_params(axis="x", rotation=0)
save_chart("02_new_york_vs_california.png")
if len(ny_ca) == 2:
    ca = ny_ca.set_index("State").loc["California"]
    ny = ny_ca.set_index("State").loc["New York"]
    print(f'California minus New York: sales {money(ca["Sales"]-ny["Sales"])}; profit {money(ca["Profit"]-ny["Profit"])}')
    print(f'Profit margins: California {ca["Margin (%)"]:.2f}%; New York {ny["Margin (%)"]:.2f}%')


## 4. Who is an outstanding customer in New York?

Rank customers by total profit and show sales as context. A customer with high sales can still generate low or negative profit.

In [ ]:
ny_customers = (df.loc[df["State"] == "New York"]
    .groupby(["Customer ID", "Customer Name"], as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")))
ny_customers["Margin (%)"] = ny_customers["Profit"] / ny_customers["Sales"] * 100
ny_customers = ny_customers.sort_values("Profit", ascending=False)
display(ny_customers.head(10))
if not ny_customers.empty:
    best = ny_customers.iloc[0]
    print(f'Highest-profit NY customer: {best["Customer Name"]} ({best["Customer ID"]}), profit {money(best["Profit"])} on sales {money(best["Sales"])}')
plt.figure(figsize=(11, 6))
sns.barplot(data=ny_customers.head(10), x="Profit", y="Customer Name", color="mediumseagreen")
plt.title("Top 10 New York Customers by Profit")
save_chart("03_new_york_top_customers.png")


## 5. Are there differences in profitability across states?

Compare profit margin (profit divided by sales), not just total profit. Low-volume states can have extreme margins, so also inspect sales volumes.

In [ ]:
by_margin = states.sort_values("Margin (%)", ascending=False)
display(by_margin[["State", "Sales", "Profit", "Margin (%)"]].head(10))
display(by_margin[["State", "Sales", "Profit", "Margin (%)"]].tail(10))
plt.figure(figsize=(12, 9))
sns.barplot(data=states.sort_values("Margin (%)"), x="Margin (%)", y="State", hue="State", palette="coolwarm", legend=False)
plt.axvline(0, color="black", lw=1)
plt.title("Profit Margin by State (Aggregate)")
save_chart("04_state_profit_margins.png")


## 6. Pareto principle — customers and profit

Sort customers by total profit. The **top 20% of customers** are selected using the ceiling of 20% of the customer count. Compare their profit contribution with the overall net profit. Because some customers may have negative profit, cumulative shares of **net** profit can exceed 100%; this is not an error. For a more stable diagnostic, also show the share of *positive* profit earned by the top 20% of customers.

In [ ]:
customers = (df.groupby(["Customer ID", "Customer Name"], as_index=False)
             .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")))
customers["Margin (%)"] = customers["Profit"] / customers["Sales"] * 100
n_customers = len(customers)
cutoff = max(1, int(np.ceil(n_customers * 0.20)))
profit_rank = customers.sort_values("Profit", ascending=False).reset_index(drop=True)
total_net_profit = profit_rank["Profit"].sum()
profit_rank["Cumulative Profit"] = profit_rank["Profit"].cumsum()
profit_rank["Customer Share (%)"] = np.arange(1, n_customers + 1) / n_customers * 100
profit_rank["Cumulative Net Profit Share (%)"] = np.where(total_net_profit != 0, profit_rank["Cumulative Profit"] / total_net_profit * 100, np.nan)
top20_net_share = profit_rank.head(cutoff)["Profit"].sum() / total_net_profit * 100 if total_net_profit != 0 else np.nan
positive_profit = profit_rank["Profit"].clip(lower=0)
positive_total = positive_profit.sum()
top20_positive_share = positive_profit.iloc[:cutoff].sum() / positive_total * 100 if positive_total else np.nan
print(f'Top {cutoff} of {n_customers} customers (20%) account for {top20_net_share:.2f}% of NET profit.')
print(f'They account for {top20_positive_share:.2f}% of POSITIVE profit (ignoring losses).')
print("Does top 20% generate at least 80% of net profit?", "Yes" if top20_net_share >= 80 else "No")
plt.figure(figsize=(10, 6))
plt.plot(profit_rank["Customer Share (%)"], profit_rank["Cumulative Net Profit Share (%)"], label="Cumulative net profit")
plt.axvline(20, color="red", ls="--", label="Top 20% customers")
plt.axhline(80, color="green", ls="--", label="80% of net profit")
plt.xlabel("Customers ranked by profit (%)")
plt.ylabel("Cumulative net profit (%)")
plt.title("Customer Profit Pareto Curve")
plt.legend()
save_chart("05_customer_profit_pareto.png")


## 7. Top 20 cities by sales and by profit

Group by **State and City together**, because city names can repeat in different states. Compare the rankings and identify cities with high sales but weak profitability.

In [ ]:
cities = (df.groupby(["State", "City"], as_index=False)
          .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")))
cities["Margin (%)"] = cities["Profit"] / cities["Sales"] * 100
cities["Location"] = cities["City"] + ", " + cities["State"]
top_cities_sales = cities.nlargest(20, "Sales")
top_cities_profit = cities.nlargest(20, "Profit")
print("TOP 20 CITIES BY SALES")
display(top_cities_sales[["Location", "Sales", "Profit", "Margin (%)"]])
print("TOP 20 CITIES BY PROFIT")
display(top_cities_profit[["Location", "Sales", "Profit", "Margin (%)"]])
print("Cities in both top-20 lists:", len(set(top_cities_sales["Location"]) & set(top_cities_profit["Location"])))
fig, axes = plt.subplots(1, 2, figsize=(17, 9))
sns.barplot(data=top_cities_sales.sort_values("Sales"), x="Sales", y="Location", ax=axes[0], color="steelblue")
sns.barplot(data=top_cities_profit.sort_values("Profit"), x="Profit", y="Location", ax=axes[1], color="seagreen")
axes[0].set_title("Top 20 Cities by Sales")
axes[1].set_title("Top 20 Cities by Profit")
save_chart("06_top_cities_sales_profit.png")
print("Highest-sales cities with negative profit:")
display(top_cities_sales.loc[top_cities_sales["Profit"] < 0, ["Location", "Sales", "Profit", "Margin (%)"]])


## 8. Top 20 customers by sales

In [ ]:
top_customers_sales = customers.nlargest(20, "Sales")
display(top_customers_sales[["Customer Name", "Sales", "Profit", "Margin (%)"]])
plt.figure(figsize=(11, 8))
sns.barplot(data=top_customers_sales.sort_values("Sales"), x="Sales", y="Customer Name", color="slateblue")
plt.title("Top 20 Customers by Sales")
save_chart("07_top_customers_sales.png")


## 9. Cumulative sales by customers — Pareto test

Since sales are positive, the cumulative sales share is straightforward to interpret. A Pareto pattern is present if the highest-spending 20% of customers account for approximately 80% or more of sales.

In [ ]:
sales_rank = customers.sort_values("Sales", ascending=False).reset_index(drop=True)
sales_rank["Customer Share (%)"] = np.arange(1, n_customers + 1) / n_customers * 100
sales_rank["Cumulative Sales Share (%)"] = sales_rank["Sales"].cumsum() / sales_rank["Sales"].sum() * 100
top20_sales_share = sales_rank.head(cutoff)["Sales"].sum() / sales_rank["Sales"].sum() * 100
print(f'Top {cutoff} of {n_customers} customers contribute {top20_sales_share:.2f}% of total sales.')
print("Does the strict 80/20 sales rule hold?", "Yes" if top20_sales_share >= 80 else "No")
plt.figure(figsize=(10, 6))
plt.plot(sales_rank["Customer Share (%)"], sales_rank["Cumulative Sales Share (%)"], label="Cumulative sales", linewidth=2)
plt.axvline(20, color="red", ls="--", label="Top 20% customers")
plt.axhline(80, color="green", ls="--", label="80% of sales")
plt.xlabel("Customers ranked by sales (%)")
plt.ylabel("Cumulative sales (%)")
plt.title("Customer Sales Pareto Curve")
plt.legend()
save_chart("08_customer_sales_pareto.png")


## 10. Product categories and monthly sales/profit trends

These diagnostics provide additional context for the marketing recommendations.

In [ ]:
categories = df.groupby("Category", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
categories["Margin (%)"] = categories["Profit"] / categories["Sales"] * 100
display(categories.sort_values("Sales", ascending=False))
monthly = df.groupby("Order Month", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
monthly["Date"] = monthly["Order Month"].dt.to_timestamp()
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(monthly["Date"], monthly["Sales"], color="steelblue")
axes[0].set_title("Monthly Sales")
axes[0].set_ylabel("Sales ($)")
axes[1].plot(monthly["Date"], monthly["Profit"], color="seagreen")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Monthly Profit")
axes[1].set_ylabel("Profit ($)")
save_chart("09_monthly_sales_profit.png")


## 11. Executive summary and marketing recommendations

Recommendations are **conditional on the observed data**. Prioritize locations with meaningful sales *and* positive profits; review discounting, product mix and costs in loss-making markets before spending more on acquisition. Sales and profits alone do not establish marketing ROI or causal impact.

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
margin = total_profit / total_sales * 100
best_sales_state = states.iloc[0]
best_profit_state = states.sort_values("Profit", ascending=False).iloc[0]
negative_states = states.loc[states["Profit"] < 0].sort_values("Profit")
priority_states = states.loc[(states["Profit"] > 0) & (states["Margin (%)"] > 0)].sort_values("Profit", ascending=False).head(5)
priority_cities = cities.loc[(cities["Profit"] > 0) & (cities["Margin (%)"] > 0)].sort_values("Profit", ascending=False).head(10)
summary = [
    "# Executive Summary — US Superstore Marketing Strategy",
    f"- Total sales: {money(total_sales)}; total profit: {money(total_profit)}; overall profit margin: {margin:.2f}%.",
    f'- Largest state by sales: {best_sales_state["State"]} ({money(best_sales_state["Sales"])}).',
    f'- Largest state by profit: {best_profit_state["State"]} ({money(best_profit_state["Profit"])}).',
    f"- Top 20% of customers generate {top20_sales_share:.2f}% of sales and {top20_net_share:.2f}% of net profit.",
    f"- The 80/20 rule for customer sales is {'supported' if top20_sales_share >= 80 else 'not supported at the strict 80% threshold'}; for customer net profit it is {'supported' if top20_net_share >= 80 else 'not supported at the strict 80% threshold'}.",
    "",
    "## Marketing priorities",
    "1. Prioritize profitable states with meaningful existing demand for retention, cross-sell and carefully tested acquisition campaigns.",
    "2. Focus city-level campaigns on high-profit markets; evaluate margin and sales together, not rankings alone.",
    "3. Build a high-value customer retention segment using profit contribution, while checking that discounts do not destroy margin.",
    "4. Review loss-making locations and product/discount policies before increasing promotional spend.",
    "5. Run controlled campaign tests and track incremental profit and customer acquisition cost before scaling budgets.",
    "",
    "## Suggested priority states (ranked by profit)",
]
summary += [f'- {r["State"]}: sales {money(r["Sales"])}, profit {money(r["Profit"])}, margin {r["Margin (%)"]:.1f}%' for _, r in priority_states.iterrows()]
summary += ["", "## Suggested priority cities (ranked by profit)"]
summary += [f'- {r["Location"]}: sales {money(r["Sales"])}, profit {money(r["Profit"])}, margin {r["Margin (%)"]:.1f}%' for _, r in priority_cities.iterrows()]
if not negative_states.empty:
    summary += ["", "## States requiring profitability review"]
    summary += [f'- {r["State"]}: profit {money(r["Profit"])}' for _, r in negative_states.iterrows()]
summary += ["", "## Methodological limitations", "- Historical observational data do not prove the effects of marketing or discounts.", "- Aggregate margins use total profit / total sales, not an unweighted mean of line-item margins.", "- Negative customer profits can make cumulative net-profit Pareto shares exceed 100%.", "- State/city rankings do not account for local market size, advertising costs, or future demand."]
report = "\n".join(summary)
print(report)
(OUT / "executive_summary.md").write_text(report, encoding="utf-8")
customers.to_csv(OUT / "customer_performance.csv", index=False)
cities.to_csv(OUT / "city_performance.csv", index=False)
categories.to_csv(OUT / "category_performance.csv", index=False)
monthly.to_csv(OUT / "monthly_sales_profit.csv", index=False)
print("\nResults saved in:", OUT.resolve())
